# Functional or object-oriented?

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*

The block closes with one problem solved in three styles: functions, dataclasses with
functions, and a class with methods; and a fourth one, numpy arrays inside an object,
for when the data are many.

**Objectives**

* compare the styles on the same problem: readability, testing, extension, speed;
* know which kind of change each style makes easy;
* choose, with reasons.

In [ ]:
import math
import random
from dataclasses import dataclass
import numpy as np

## 1. The problem

A calorimeter gives, for each hit, a channel number and an ADC count. We must:

1. **calibrate**: energy (MeV) $= a \, \mathrm{ADC} + b$;
2. **select** the hits above a threshold;
3. compute the **total** and the **mean** energy of the selected hits.

In [ ]:
rng = random.Random(1)
raw = [(rng.randrange(64), rng.randrange(4096)) for _ in range(1000)]   # (channel, adc)
SLOPE, OFFSET, THRESHOLD = 0.05, 0.2, 50.                             # MeV/ADC, MeV, MeV
print(raw[:3])

## 2. Three styles

### 2.1 Functional: pure functions on plain data

In [ ]:
def calibrate(hits, slope, offset):
    return [(ch, slope * adc + offset) for ch, adc in hits]

def select(hits, threshold):
    return [(ch, e) for ch, e in hits if e > threshold]

def total_and_mean(hits):
    energies = [e for _, e in hits]
    return sum(energies), sum(energies) / len(energies)

print(total_and_mean(select(calibrate(raw, SLOPE, OFFSET), THRESHOLD)))

# a test needs only an input and the expected output
assert total_and_mean([(0, 10.), (1, 30.)]) == (40., 20.)

### 2.2 Data classes and functions

The same functions, but the data have names: `hit.energy` instead of `hit[1]`.

In [ ]:
@dataclass(frozen=True)
class Hit:
    channel: int
    energy: float

def calibrate_hits(raw, slope, offset):
    return [Hit(ch, slope * adc + offset) for ch, adc in raw]

def select_hits(hits, threshold):
    return [h for h in hits if h.energy > threshold]

def summary(hits):
    total = sum(h.energy for h in hits)
    return total, total / len(hits)

print(summary(select_hits(calibrate_hits(raw, SLOPE, OFFSET), THRESHOLD)))

### 2.3 A class with methods

The data and the operations together, in an object:

In [ ]:
class Sample:
    """Calibrated calorimeter hits."""

    def __init__(self, channels, energies):
        self.channels = list(channels)
        self.energies = list(energies)

    @classmethod
    def from_raw(cls, raw, slope, offset):
        return cls([ch for ch, _ in raw], [slope * adc + offset for _, adc in raw])

    def select(self, threshold):
        """A new Sample with the hits above threshold."""
        kept = [(ch, e) for ch, e in zip(self.channels, self.energies) if e > threshold]
        return Sample([ch for ch, _ in kept], [e for _, e in kept])

    def total(self):
        return sum(self.energies)

    def mean(self):
        return self.total() / len(self.energies)

s = Sample.from_raw(raw, SLOPE, OFFSET).select(THRESHOLD)
print(s.total(), s.mean())

t = Sample([0, 1], [10., 30.])
assert (t.total(), t.mean()) == (40., 20.)

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Which one would you keep?</b></p>

Compare the three versions:

1. Which one is the easiest to **read**? To **test**?
2. Which one would you rather extend if a second detector arrives, with a **different
   calibration** (quadratic, per channel)?
3. And if instead you need a **new operation** on the hits (an energy-weighted mean
   channel, a histogram)?

</div>

<details>
<summary><b>Solution</b></summary>

1. The functional versions are the easiest to test: each function is pure, and a test is
   one input and one output. The data class version reads best (`h.energy`).
2. A new **calibration** is a new function (functional) or a new constructor or class
   (object-oriented): both are easy here, because calibration is separate from the
   analysis. The workshop of section 4 does exactly this.
3. A new **operation** (an energy-weighted mean channel) is one new function in the
   functional style, and touches nothing else; in the object-oriented style it is a new
   method in `Sample`, and in every class that offers the same interface.

The styles really differ when the code works with **several kinds** of things: a new kind
over fixed operations is one new class in the object-oriented style, and the code that
uses it does not change; in the functional style every function with an `if/elif` on the
kind must be edited. A new operation over fixed kinds is the opposite. This trade-off has
a name, the *expression problem*. Real codes mix both styles.

</details>

## 3. And with many hits?

All three versions create one Python object (a tuple, a `Hit`, a float) per hit. With
$10^7$ hits that is slow. The fourth style keeps the **structure** in an object and the
**numbers** in numpy arrays.

In [ ]:
class ArraySample:
    """Calibrated calorimeter hits, stored as numpy arrays."""

    def __init__(self, channels, energies):
        self.channels = np.asarray(channels)
        self.energies = np.asarray(energies, dtype=float)

    @classmethod
    def from_raw(cls, raw, slope, offset):
        channels, adc = np.asarray(raw).T
        return cls(channels, slope * adc + offset)

    def select(self, threshold):
        mask = self.energies > threshold
        return ArraySample(self.channels[mask], self.energies[mask])

    def total(self):
        return self.energies.sum()

    def mean(self):
        return self.energies.mean()

a = ArraySample.from_raw(raw, SLOPE, OFFSET).select(THRESHOLD)
print(a.total(), a.mean())

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — How much faster?</b></p>

With $10^6$ hits, **bet** on the speed-up of `ArraySample` over the functional version of
section 2.1: 2, 10 or 100 times?

</div>

In [ ]:
big = [(rng.randrange(64), rng.randrange(4096)) for _ in range(1_000_000)]
big_array = np.array(big)

%timeit total_and_mean(select(calibrate(big, SLOPE, OFFSET), THRESHOLD))
%timeit ArraySample(big_array[:, 0], SLOPE * big_array[:, 1] + OFFSET).select(THRESHOLD).mean()

<details>
<summary><b>Solution</b></summary>

Typically between 10 and 100 times, depending on the machine. But be fair: the array
version starts from an array; converting the list costs about as much as the functional
pipeline. Keep the data in arrays from the start.

Note that `ArraySample` is functional inside: `select` returns a **new** object and never
modifies `self`, and numpy expressions are `map` and `filter` over whole arrays. Objects
for the structure, arrays for the numbers, pure operations.

</details>

## 4. Practice

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (in pairs) — A new requirement</b></p>

The calibration changes: each channel has its own slope, given by a list `slopes` of 64
values. In pairs: one person adapts the functional version (2.1), the other the class
(2.3 or 3). **First** write one test for your version (for example: the mean of the
selected energies is above the threshold), then change the calibration, and check that
both versions give the same total and mean.

How many lines did each one change? Where did the change go?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
slopes = [0.05 * (1 + 0.01 * (ch % 5)) for ch in range(64)]

# functional: a new calibration function; select and total_and_mean do not change
def calibrate_per_channel(hits, slopes, offset):
    return [(ch, slopes[ch] * adc + offset) for ch, adc in hits]

total, mean = total_and_mean(select(calibrate_per_channel(raw, slopes, OFFSET), THRESHOLD))
assert mean > THRESHOLD

# object-oriented (arrays): a new alternative constructor; select, total, mean unchanged
def from_raw_per_channel(raw, slopes, offset):
    channels, adc = np.asarray(raw).T
    return ArraySample(channels, np.asarray(slopes)[channels] * adc + offset)

a = from_raw_per_channel(raw, slopes, OFFSET).select(THRESHOLD)
assert a.mean() > THRESHOLD
assert math.isclose(a.mean(), mean) and math.isclose(a.total(), total)
```

In both cases the change stays in one place, because both designs separate calibration
from analysis. The design matters more than the paradigm.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Review this design</b></p>

A student wrote this. What would you change?

```python
class Analysis:
    def __init__(self, filename):
        self.data = open(filename).read().split()
        self.result = None

    def run(self):
        total = 0
        for i in range(len(self.data)):
            total = total + float(self.data[i]) * 0.05 + 0.2
        self.result = total
        print('total', self.result)
```

</div>

<details>
<summary><b>Solution</b></summary>

* The constructor reads a file and never closes it; reading belongs in a function (or a
  `classmethod`) with `with open(...)`.
* `run` mixes calibration, sum and printing, and returns nothing: it cannot be tested.
* `self.result` is state that exists only after `run`: an object that is half-built.
* `print` inside the computation: return the value, print at the edge.
* Magic numbers, `range(len(...))`.

A functional core with pure, testable pieces, and an object (or none) only for what must
be kept together:

```python
def read_adc(path):
    with open(path) as f:
        return [float(x) for x in f.read().split()]

def calibrate_adc(adc, slope=0.05, offset=0.2):
    return [slope * a + offset for a in adc]

def total_energy(path):
    return sum(calibrate_adc(read_adc(path)))
```

</details>

And if the hits do not fit in memory, the functional style becomes a lazy pipeline of
[generators](./about_iterators.ipynb), one hit at a time:

In [ ]:
def stream_hits(n, seed=2):
    r = random.Random(seed)
    for _ in range(n):
        yield r.randrange(64), r.randrange(4096)

energies = (SLOPE * adc + OFFSET for _, adc in stream_hits(10**6))
print(sum(e for e in energies if e > THRESHOLD))

***[+] Lookout.*** *Functional core, imperative shell*: keep the computations pure, and the reading, writing
and plotting at the edges. Other languages take other sides: Julia builds on *multiple
dispatch* (functions that choose their method by the types of all the arguments: one
answer to the expression problem), Haskell
is purely functional, and C++ mixes both, like Python.

## 5. Closing the block

* **Functions**: small, pure, tested. They are the unit of everything else.
* **Iterators and generators**: data that flows, one item at a time.
* **Classes**: when data and behaviour belong together, when invariants must hold, and
  when several implementations share one interface.
* **Tests** for all of it, written first when you can.
* For many numbers: **arrays**, inside a few objects, with pure operations.

<details>
<summary><b>[i] Cheat sheet — Which style?</b></summary>

| If… | then… |
|:--|:--|
| it computes a value from its inputs | a **pure function** |
| it is data with names, little behaviour | a `@dataclass` (`frozen=True`) |
| there is state and behaviour that belong together | a **class** |
| there will be several implementations of one idea | a class per implementation, one **interface** |
| you will add many new operations | functions over simple data tend to be easier |
| you will add many new kinds of things | classes with a common interface tend to be easier |
| there are millions of identical items | **numpy arrays**, inside a few objects |
| the data come as a stream, or do not fit in memory | **generators** and a pipeline |

</details>

### Check yourself

1. Why is a pure function easier to test than an object with hidden state?
2. Which change is easy in the functional style, and which in the object-oriented one?
3. Why is "one object per hit" slow, and what is the alternative?
4. Give one case where you would write a class, and one where you would not.

<details>
<summary><b>Solutions</b></summary>

1. Its result depends only on its arguments: no state to prepare, nothing else to check.
2. New operations in the functional style; new kinds of things in the object-oriented one.
3. Each object costs memory and a Python-level loop; keep the numbers in numpy arrays
   inside a few objects.
4. A class: a detector model with parameters, state and several methods, with several
   implementations. Not a class: a unit conversion (a function) or a fit result (a
   dataclass).

</details>

End of the block. Back to the [introduction](./python_intro.ipynb).